In [1]:
import torch 
import torch.nn as nn

import numpy as np

In [13]:
import ast
import random

with open("Data/grok5000.txt", "r", encoding="utf-8") as file:
    number_lines = sum(1 for _ in file)

    data = np.empty(number_lines, dtype=tuple)
    file.seek(0)  

    for i, line in enumerate(file):
        text, label_list = ast.literal_eval(line.strip())
        data[i] = (text, torch.tensor(label_list, dtype=torch.long))


random.seed(42)
random.shuffle(data)

split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

Train: 4500, Val: 500


In [11]:
import importlib
from Model.parser import Parser

parser = Parser()

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
parser.to(device)

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, parser.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

num_epochs = 1
grad_clip = 5.0

# ============================================================
# Training Loop
# ============================================================
from tqdm import tqdm

for epoch in range(num_epochs):
    parser.train()
    random.shuffle(train_data)
    epoch_loss, correct, total = 0.0, 0, 0

    train_bar = tqdm(train_data, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    for text, targets in train_bar:
        targets = targets.to(device)

        optimizer.zero_grad()

        ids, mask, _ = parser.prepare_input(text)
        number_predictions = parser(ids, mask)

        if number_predictions.size(0) != targets.size(0):
            continue  # sollte nach den Fixes oben nicht mehr passieren

        loss = criterion(number_predictions, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parser.parameters(), grad_clip)
        optimizer.step()

        epoch_loss += loss.item()
        correct += (number_predictions.argmax(dim=-1) == targets).sum().item()
        total += targets.size(0)

        # Statusbar live aktualisieren
        train_bar.set_postfix(
            loss=epoch_loss / max(total, 1),
            acc=correct / total if total > 0 else 0.0
        )

    # ---- Validation ----
    parser.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_bar = tqdm(val_data, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for text, targets in val_bar:
            targets = targets.to(device)
            
            
            ids, mask, _ = parser.prepare_input(text)
            number_predictions = parser(ids, mask)

            if number_predictions.size(0) != targets.size(0):
                continue
            loss = criterion(number_predictions, targets)
            val_loss += loss.item()
            val_correct += (number_predictions.argmax(dim=-1) == targets).sum().item()
            val_total += targets.size(0)

            val_bar.set_postfix(
                loss=val_loss / max(val_total, 1),
                acc=val_correct / val_total if val_total > 0 else 0.0
            )

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {epoch_loss/len(train_data):.4f} Acc: {1 if total == 0 else correct/total:.4f} | "
          f"Val Loss: {val_loss/len(val_data):.4f} Acc: {1 if val_total == 0 else val_correct/val_total:.4f}")

Epoch 1/1 | Train Loss: 0.0003 Acc: 0.0833 | Val Loss: 0.0000 Acc: 1.0000


In [15]:
def evalModel(model, val_data):
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_bar = tqdm(val_data, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for text, targets in val_bar:
            targets = targets.to(device)
            
            
            ids, mask, _ = model.prepare_input(text)
            number_predictions = model(ids, mask)
            
            if number_predictions.size(0) != targets.size(0):
                continue
            loss = criterion(number_predictions, targets)
            val_loss += loss.item()
            val_correct += (number_predictions.argmax(dim=-1) == targets).sum().item()
            val_total += targets.size(0)

            val_bar.set_postfix(
                loss=val_loss / max(val_total, 1),
                acc=val_correct / val_total if val_total > 0 else 0.0
            )
    print(f"Val Loss: {val_loss/len(val_data):.4f} Acc: {1 if val_total == 0 else val_correct/val_total:.4f}")
    

evalModel(parser, val_data)

# save weights in file
# torch.save(parser.state_dict(), "trainedModels/parser.pth")

Val Loss: 0.0000 Acc: 1.0000


In [16]:

dummy_ids, dummy_mask, _ = parser.prepare_input("Test 123")

torch.onnx.export(
    parser,
    (dummy_ids, dummy_mask),
    "trainedModels/parser.onnx",
    input_names=["token_ids", "number_mask"],
    output_names=["predictions"],
    dynamic_axes={
        "token_ids": {0: "seq_len"},
        "number_mask": {0: "seq_len"},
        "predictions": {0: "seq_len"},
    },
    dynamo=False,   # expliziter Rückfall auf den älteren, stabileren Pfad
)

print("ONNX-Export erfolgreich: parser.onnx")

C:\Users\Jan\AppData\Local\Temp\ipykernel_17500\3062472661.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX-Export erfolgreich: parser.onnx
